# Solar Power Generation Data Analysis

After the data understanding we want to extract some insight from the singles inverters, looking for anomalies. 

Focused analysis following your 6 steps:
1. Load data
2. Data preparation  
3. Analyze inverter sizes (average AC_POWER)
4. Conversion efficiency analysis 
5. DC power normalization, visual detection


## Step 1: Load Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load datasets
generation_df = pd.read_csv('../data/Plant_2_Generation_Data.csv')
weather_df = pd.read_csv('../data/Plant_2_Weather_Sensor_Data.csv')

# Convert DATE_TIME
generation_df['DATE_TIME'] = pd.to_datetime(generation_df['DATE_TIME'])
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])

# Create DAY and TIME columns
generation_df['DAY'] = generation_df['DATE_TIME'].dt.date
generation_df['DAYTIME'] = generation_df['DATE_TIME'].dt.time

weather_df['DAY'] = weather_df['DATE_TIME'].dt.date
weather_df['DAYTIME'] = weather_df['DATE_TIME'].dt.time

print(f"Generation data: {generation_df.shape}")
print(f"Unique inverters: {generation_df['SOURCE_KEY'].nunique()}")
print(f"Date range: {generation_df['DATE_TIME'].min()} to {generation_df['DATE_TIME'].max()}")

## Step 2: Data Preparation

With our data understandig we discover that we have two cluster of inverter with different data completness. We want to perform the analysis on the cluster with a better data quality.

In [ ]:
# Calculate data completeness
total_sample_per_day = 96
source_dataquality = generation_df.groupby(['DAY','SOURCE_KEY'])[['DAYTIME']].count().rename(columns={'DAYTIME': 'sample_completeness'})
source_dataquality['sample_completeness'] = round(source_dataquality['sample_completeness'] / total_sample_per_day * 100, 2)

# Create heatmap data
heatmap_data = source_dataquality.reset_index().pivot(index='SOURCE_KEY', columns='DAY', values='sample_completeness')

# Select SOURCE_KEYs with more than 98% average completeness
completeness_threshold = 98.0
source_avg_completeness = heatmap_data.mean(axis=1)
high_quality_sources = list(source_avg_completeness[source_avg_completeness > completeness_threshold].index)

print(f"High-quality sources: {len(high_quality_sources)} out of {len(source_avg_completeness)}")
print(f"Selected: {high_quality_sources}")

# Filter data
generation_df = generation_df[generation_df.SOURCE_KEY.isin(high_quality_sources)]
print(f"Filtered data shape: {generation_df.shape}")

## Step 3: Analyze Inverter Size - Average AC Power




In [ ]:
# Calculate average AC power per inverter (only when generating)
inverter_ac_stats = generation_df[generation_df['AC_POWER'] > 0].groupby('SOURCE_KEY')['AC_POWER'].agg(['mean', 'max', 'std', 'count']).round(2)

# Sort inverters by mean ac power
inverter_ac_stats.sort_values(by='mean', ascending=False, inplace=True)

print("AC Power Statistics by Inverter:")
print(inverter_ac_stats)

# Plot
plt.figure(figsize=(12, 6))
inverter_ac_stats['mean'].plot(kind='bar', edgecolor='black')
plt.title('Average AC Power by Inverter', fontweight='bold')
plt.xlabel('Inverter (SOURCE_KEY)')
plt.ylabel('Average AC Power (kW)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSummary - Average AC Power:")
print(f"Overall average: {inverter_ac_stats['mean'].mean():.2f} kW")
print(f"Range: {inverter_ac_stats['mean'].min():.2f} to {inverter_ac_stats['mean'].max():.2f} kW")
print(f"Std deviation: {inverter_ac_stats['mean'].std():.2f} kW")

## Step 4: Conversion Efficiency Analysis

In [ ]:
# Calculate conversion efficiency
generation_df['conversion_efficiency'] = np.where(
    generation_df['DC_POWER'] > 0,
    (generation_df['AC_POWER'] / generation_df['DC_POWER']) * 100,
    np.nan
)

# Stats per inverter
efficiency_stats = generation_df.groupby('SOURCE_KEY')['conversion_efficiency'].agg(['mean', 'std', 'count']).round(2)

# Sort by mean efficiency
efficiency_stats.sort_values(by='mean', ascending=False, inplace=True)

print("Conversion Efficiency by Inverter:")
print(efficiency_stats)

# Plot efficiency
plt.figure(figsize=(12, 6))
efficiency_stats['mean'].plot(kind='bar', color='lightgreen', edgecolor='black')
plt.title('Average Conversion Efficiency by Inverter', fontweight='bold')
plt.xlabel('Inverter (SOURCE_KEY)')
plt.ylabel('Conversion Efficiency (%)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nEfficiency Summary:")
print(f"Average: {efficiency_stats['mean'].mean():.2f}%")
print(f"Range: {efficiency_stats['mean'].min():.2f}% to {efficiency_stats['mean'].max():.2f}%")

Average conversion efficiency is very similar for all the inverters

In [ ]:
plt.figure(figsize=(16, 6))
sns.boxplot(x='SOURCE_KEY', y='conversion_efficiency', data=generation_df)
plt.title('Inverter Efficiency Distribution')
plt.xticks(rotation=45)
plt.show()

I want to check what happend in that particular outlier on inverter "xoJJ8DcxJEcupym"

In [ ]:
generation_df.sort_values(by=['conversion_efficiency']).head(1)

In [ ]:
# Filter data for specific source key and date
target_source = 'xoJJ8DcxJEcupym'
target_date = pd.to_datetime('2020-05-27').date()

# Filter the data
filtered_data = generation_df[
    (generation_df['SOURCE_KEY'] == target_source) & 
    (generation_df['DAY'] == target_date)
].copy()

filtered_data = filtered_data.sort_values('DATE_TIME')
    
# Create the plot
plt.figure(figsize=(14, 6))
plt.plot(filtered_data['DATE_TIME'], filtered_data['DC_POWER'], 
            color='blue', linewidth=2, marker='o', markersize=3, label='DC Power')
plt.plot(filtered_data['DATE_TIME'], filtered_data['AC_POWER'], 
            color='red', linewidth=2, marker='o', markersize=3, label='AC Power')
plt.title(f'DC and AC Power Comparison for {target_source} on {target_date}', fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Power (kW)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.show()



We found an interruption in the generation of this inverter, is it due to curtailment, clipping or is it a malfunction?


## Step 5: DC Power Normalization, visual detection

The irradiation on the plant is the same, so we can expect the same DC_POWER curve for every inverter. Let's have a look at them with the time-based analysis

First, we have to normalize the data to compare the curve, a simple way is to compute the mean DC_POWER of each inverter and scale it's power by this value.

In [ ]:
# Merge with weather data
weather_df['DAY'] = weather_df['DATE_TIME'].dt.date
weather_df['DAYTIME'] = weather_df['DATE_TIME'].dt.time
print(f"Dataframe shape before merge: {generation_df.shape}")
merged_data = pd.merge(generation_df, weather_df[['DATE_TIME', 'IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']], on='DATE_TIME', how='inner')
print(f"Dataframe shape after merge: {merged_data.shape}")

# Calculate average DC power per inverter (for normalization)
inverter_avg_dc = generation_df[generation_df['DC_POWER'] > 0].groupby('SOURCE_KEY')['DC_POWER'].mean().sort_values(ascending=False)

# Normalize DC power
def normalize_dc(row):
    if row['DC_POWER'] > 0 and row['SOURCE_KEY'] in inverter_avg_dc:
        return row['DC_POWER'] / inverter_avg_dc[row['SOURCE_KEY']]
    return np.nan

merged_data['normalized_dc_power'] = merged_data.apply(normalize_dc, axis=1)

Now is possible to visualize the different shapes

In [ ]:
# Compare normalized performance by hour
merged_data['HOUR'] = merged_data['DATE_TIME'].dt.hour
hourly_perf = merged_data.groupby(['HOUR', 'SOURCE_KEY'])['normalized_dc_power'].mean().reset_index()
hourly_pivot = hourly_perf.pivot(index='HOUR', columns='SOURCE_KEY', values='normalized_dc_power')

# Plot normalized performance
plt.figure(figsize=(14, 8))
for inverter in high_quality_sources:
    if inverter in hourly_pivot.columns:
        plt.plot(hourly_pivot.index, hourly_pivot[inverter], marker='o', label=inverter, alpha=0.7)

plt.title('Normalized DC Power by Hour (Relative to Inverter Average)', fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Normalized DC Power')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Performance variation by hour
perf_variation = hourly_pivot.std(axis=1)
print(f"\nHours with highest performance variation:")
print(perf_variation.sort_values(ascending=False).head())

##### Connecting Variation to Field Action: Shadowing and Maintenance

The analysis of performance variation by hour provides a crucial link between data and physical asset management.

The observation that the **highest performance variation occurs during the early and late hours** of the day strongly suggests a physical, non-weather-related issue, such as **Shadowing:** 

    Fixed objects (trees, adjacent buildings, other rows of panels) may be casting shadows on certain inverters in a non-uniform way, affecting different strings at different times.


This pattern demonstrates how **Data Analytics can directly guide Field Inspections**:

* Instead of checking every panel, the operations team can prioritize an **on-site inspection** of the areas covered by the inverters showing the highest deviation in those specific hours, turning a statistical outlier into a concrete, actionable maintenance task.